In [0]:
# Loads the Plan Attributes and Rates CSVs into Delta tables.
# - Every original CMS column is kept as exact text, so accuracy can be proven.
# - Number-like columns also get a "<name>_num" DECIMAL partner, so numeric filters work.
# - Each table only saves if every year's loaded row count equals source_manifest.

import os
import re
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SCHEMA = "<your_catalog>.<your_schema>"
MANIFEST_TABLE = f"{SCHEMA}.source_manifest"
NUMERIC_PROFILE_TABLE = f"{SCHEMA}.numeric_profile"
DATASETS_TO_LOAD = ["plan_attributes", "rates"]  # benefits_and_cost_sharing is a stretch goal
SAFE_COLUMN_NAME = re.compile(r"^[a-z][a-z0-9_]*$")
NEVER_NUMERIC = {"business_year", "plan_year", "source_file"}  # labels, not quantities


def to_snake_case(cms_column_name: str) -> str:
    """'HIOSProductId' -> 'hios_product_id'; 'MEHBInnTier1IndividualMOOP' -> 'mehb_inn_tier1_individual_moop'."""
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", cms_column_name.strip())
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower()


def read_cms_csv(csv_path: str) -> DataFrame:
    # multiLine handles quoted text with line breaks; FAILFAST stops on malformed rows
    # instead of silently dropping them (a silent drop would break row-count verification).
    return (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
            .option("inferSchema", False).option("mode", "FAILFAST").csv(csv_path))


def rename_to_snake_case(df: DataFrame) -> DataFrame:
    new_names = [to_snake_case(c) for c in df.columns]
    bad = [n for n in new_names if not SAFE_COLUMN_NAME.match(n)]
    dupes = sorted({n for n in new_names if new_names.count(n) > 1})
    if bad or dupes:
        raise ValueError(f"Column rename problem. Invalid: {bad}  Duplicates: {dupes}")
    return df.toDF(*new_names)


def as_number(column_name: str):
    """'$2,900 ' -> 2900.0000, '73.22%' -> 73.2200, 'Not Applicable' -> NULL (never an error)."""
    return F.expr(f"try_cast(regexp_replace(`{column_name}`, '[$,% ]', '') AS DECIMAL(18,4))")


def profile_numeric_columns(df: DataFrame, dataset: str) -> list[dict]:
    """A column is numeric if some values parse AND every value containing a digit parses.
    So 'Not Applicable' is allowed (no digits) but '0-14' or '64 and over' make it text-only."""
    candidates = [c for c in df.columns if c not in NEVER_NUMERIC and not c.endswith("_id")]
    checks = []
    for c in candidates:
        has_digit = F.col(c).rlike("[0-9]")
        checks += [
            F.count(as_number(c)).alias(f"ok__{c}"),
            F.sum(F.when(has_digit & as_number(c).isNull(), 1).otherwise(0)).alias(f"bad__{c}"),
            F.sum(F.when(F.col(c).isNotNull() & ~has_digit, 1).otherwise(0)).alias(f"text__{c}"),
            F.max(F.col(c).contains("$").cast("int")).alias(f"money__{c}"),
            F.max(F.col(c).contains("%").cast("int")).alias(f"pct__{c}"),
        ]
    stats = df.agg(*checks).collect()[0].asDict()
    profile = []
    for c in candidates:
        if stats[f"ok__{c}"] > 0 and (stats[f"bad__{c}"] or 0) == 0:
            unit = "money" if stats[f"money__{c}"] else "percent" if stats[f"pct__{c}"] else "number"
            profile.append({"dataset": dataset, "variable": c, "numeric_column": f"{c}_num", "unit": unit,
                            "numeric_values": int(stats[f"ok__{c}"]), "text_only_values": int(stats[f"text__{c}"] or 0)})
    return profile


summary, all_profiles = [], []
for dataset in DATASETS_TO_LOAD:
    manifest = spark.table(MANIFEST_TABLE).filter(F.col("dataset") == dataset).orderBy("plan_year").collect()
    yearly_frames, counts_match = [], True
    for entry in manifest:
        df = rename_to_snake_case(read_cms_csv(entry.csv_file))
        df = (df.withColumn("plan_year", F.lit(entry.plan_year).cast("int"))
                .withColumn("source_file", F.lit(f"{entry.plan_year}/{os.path.basename(entry.csv_file)}")))
        loaded_rows = df.count()
        result = "MATCH" if loaded_rows == entry.row_count else "MISMATCH"
        counts_match = counts_match and result == "MATCH"
        summary.append((dataset, entry.plan_year, entry.row_count, loaded_rows, result))
        print(f"{dataset} {entry.plan_year}: CMS file {entry.row_count:,} rows | loaded {loaded_rows:,} | {result}")
        yearly_frames.append(df)

    if not counts_match:
        print(f"   NOT saving {SCHEMA}.{dataset} because a row count did not match.")
        continue

    # unionByName lines columns up by name; a column missing in one year is empty for that year.
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), yearly_frames)
    profile = profile_numeric_columns(combined, dataset)
    existing = set(combined.columns)
    clashes = [p["numeric_column"] for p in profile if p["numeric_column"] in existing]
    if clashes:
        raise ValueError(f"Numeric column names already exist: {clashes}")
    combined = combined.select("*", *[as_number(p["variable"]).alias(p["numeric_column"]) for p in profile])

    combined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.{dataset}")
    spark.sql(f"COMMENT ON TABLE {SCHEMA}.{dataset} IS 'CMS Marketplace QHP {dataset} PUF, plan years stacked. "
              f"Original columns are exact CMS text; *_num columns are numeric versions for filtering. Public data.'")
    all_profiles += profile
    print(f"   Saved {SCHEMA}.{dataset}: {len(combined.columns)} columns, {len(profile)} with a numeric version")

if all_profiles:
    (spark.createDataFrame(all_profiles)
        .select("dataset", "variable", "numeric_column", "unit", "numeric_values", "text_only_values")
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(NUMERIC_PROFILE_TABLE))
    print(f"Saved {len(all_profiles)} rows to {NUMERIC_PROFILE_TABLE}")

display(spark.createDataFrame(summary, "dataset string, plan_year int, cms_file_rows long, loaded_rows long, result string"))